In [ ]:
from pathlib import Path
import json
import math
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
from tqdm.auto import tqdm

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    #wwwwwwwwwwwwwwww


In [ ]:
# configuration

PROJECT_ROOT = Path(r"C:\Emotion-Recognition")
DATA_ROOT = PROJECT_ROOT / "archive"
TRAIN_DIR = DATA_ROOT / "train"
TEST_DIR = DATA_ROOT / "test"
OUTPUT_DIR = PROJECT_ROOT / "vit_runs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "vit_tiny_patch16_224"
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.08
VAL_SPLIT = 0.15
SEED = 42
NUM_WORKERS = 0  .

DROP_RATE = 0.15
DROP_PATH_RATE = 0.10
LABEL_SMOOTHING = 0.12
EARLY_STOPPING_PATIENCE = 10
MIN_DELTA = 0.001

MODEL_PATH = OUTPUT_DIR / "vit_fer2013_6class_regularized_best.pth"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


seed_everything(SEED)

In [ ]:
assert TRAIN_DIR.exists(), f"Train folder not found: {TRAIN_DIR}"
assert TEST_DIR.exists(), f"Test folder not found: {TEST_DIR}"

train_folders = sorted([p.name for p in TRAIN_DIR.iterdir() if p.is_dir()])
test_folders = sorted([p.name for p in TEST_DIR.iterdir() if p.is_dir()])

print("Train folders:", train_folders)
print("Test folders:", test_folders)
assert train_folders == test_folders, "Train and test class folders must match."
assert "disgust" not in train_folders, "Disgust still exists in the dataset folders."
print("Number of classes:", len(train_folders))

In [ ]:
def count_images(folder):
    rows = []
    for class_dir in sorted([p for p in folder.iterdir() if p.is_dir()]):
        count = sum(1 for p in class_dir.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"})
        rows.append({"class": class_dir.name, "count": count})
    return pd.DataFrame(rows)


train_counts = count_images(TRAIN_DIR)
test_counts = count_images(TEST_DIR)

display(train_counts)
display(test_counts)

In [ ]:
# stronger preprocessing and augmentation

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.78, 1.0), ratio=(0.90, 1.10)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(degrees=0, translate=(0.10, 0.10), scale=(0.90, 1.10)),
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.10),
    transforms.RandomGrayscale(p=0.10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.15), ratio=(0.3, 3.3)),
])

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
base_train_dataset = datasets.ImageFolder(TRAIN_DIR)
base_test_dataset = datasets.ImageFolder(TEST_DIR)

class_names = base_train_dataset.classes
display_class_names = ["surprised" if name == "suprised" else name for name in class_names]
num_classes = len(class_names)

print("Raw class names:", class_names)
print("Display class names:", display_class_names)

assert base_train_dataset.class_to_idx == base_test_dataset.class_to_idx
print("Class to index:", base_train_dataset.class_to_idx)

In [ ]:
class EmotionFolderDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples = list(samples)
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        image_path, label = self.samples[index]
        image = Image.open(image_path).convert("RGB")
        image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.long)


train_indices, val_indices = train_test_split(
    np.arange(len(base_train_dataset.samples)),
    test_size=VAL_SPLIT,
    random_state=SEED,
    stratify=base_train_dataset.targets,
)

train_samples = [base_train_dataset.samples[i] for i in train_indices]
val_samples = [base_train_dataset.samples[i] for i in val_indices]
test_samples = base_test_dataset.samples

train_dataset = EmotionFolderDataset(train_samples, train_transform)
val_dataset = EmotionFolderDataset(val_samples, eval_transform)
test_dataset = EmotionFolderDataset(test_samples, eval_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

In [ ]:
def denormalize(tensor):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return torch.clamp(tensor.cpu() * std + mean, 0, 1)


images, labels = next(iter(train_loader))
plt.figure(figsize=(10, 6))
for i in range(min(12, len(images))):
    plt.subplot(3, 4, i + 1)
    plt.imshow(denormalize(images[i]).permute(1, 2, 0))
    plt.title(display_class_names[labels[i].item()])
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = timm.create_model(
    MODEL_NAME,
    pretrained=True,
    num_classes=num_classes,
    drop_rate=DROP_RATE,
    drop_path_rate=DROP_PATH_RATE,
).to(device)

class_counts = np.bincount([label for _, label in train_samples], minlength=num_classes).astype(np.float32)
class_weights = class_counts.sum() / np.maximum(class_counts, 1.0)
class_weights = class_weights / class_weights.mean()
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32, device=device)

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor, label_smoothing=LABEL_SMOOTHING)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda")

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Model:", MODEL_NAME)
print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)
print("Drop rate:", DROP_RATE)
print("Drop path rate:", DROP_PATH_RATE)
print("Label smoothing:", LABEL_SMOOTHING)
print("Class weights:", dict(zip(display_class_names, class_weights.round(3))))

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train_one_epoch(model, loader):
    model.train()
    total_loss = 0.0
    all_preds = []
    all_targets = []

    for images, labels in tqdm(loader, desc="train", leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=device.type == "cuda"):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * images.size(0)
        all_preds.extend(outputs.argmax(1).detach().cpu().numpy())
        all_targets.extend(labels.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_targets, all_preds)
    return avg_loss, acc


@torch.no_grad()
def evaluate(model, loader, desc="eval"):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_targets = []

    for images, labels in tqdm(loader, desc=desc, leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        total_loss += loss.item() * images.size(0)
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_targets.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_targets, all_preds)
    macro_f1 = f1_score(all_targets, all_preds, average="macro")
    return avg_loss, acc, macro_f1, all_targets, all_preds

In [ ]:
best_val_f1 = 0.0
best_val_acc = 0.0
best_epoch = 0
epochs_without_improvement = 0
history = []

for epoch in range(1, EPOCHS + 1):
    start = time.time()
    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_loss, val_acc, val_f1, _, _ = evaluate(model, val_loader, desc="val")
    scheduler.step()

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "val_macro_f1": val_f1,
        "seconds": time.time() - start,
    }
    history.append(row)

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train_acc={train_acc:.4f} | val_acc={val_acc:.4f} | "
        f"val_f1={val_f1:.4f} | time={row['seconds']:.1f}s"
    )

    # Macro F1 is better than accuracy for this imbalanced dataset.
    if val_f1 > best_val_f1 + MIN_DELTA:
        best_val_f1 = val_f1
        best_val_acc = val_acc
        best_epoch = epoch
        epochs_without_improvement = 0
        torch.save(
            {
                "model_state": model.state_dict(),
                "model_name": MODEL_NAME,
                "class_names": class_names,
                "display_class_names": display_class_names,
                "class_to_idx": base_train_dataset.class_to_idx,
                "img_size": IMG_SIZE,
                "best_val_acc": best_val_acc,
                "best_val_macro_f1": best_val_f1,
                "best_epoch": best_epoch,
                "regularization": {
                    "drop_rate": DROP_RATE,
                    "drop_path_rate": DROP_PATH_RATE,
                    "label_smoothing": LABEL_SMOOTHING,
                    "weight_decay": WEIGHT_DECAY,
                    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
                },
            },
            MODEL_PATH,
        )
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(f"Early stopping at epoch {epoch}. Best epoch: {best_epoch}")
        break

history_df = pd.DataFrame(history)
history_df.to_csv(OUTPUT_DIR / "vit_regularized_training_history.csv", index=False)
print("Best validation accuracy:", best_val_acc)
print("Best validation macro F1:", best_val_f1)
print("Best epoch:", best_epoch)
print("Saved best model to:", MODEL_PATH)

In [ ]:
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(history_df["epoch"], history_df["train_acc"], label="train")
plt.plot(history_df["epoch"], history_df["val_acc"], label="validation")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.title("Accuracy")

plt.subplot(1, 2, 2)
plt.plot(history_df["epoch"], history_df["train_loss"], label="train")
plt.plot(history_df["epoch"], history_df["val_loss"], label="validation")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("Loss")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "vit_regularized_training_curves.png", dpi=180)
plt.show()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(checkpoint["model_state"])
model.eval()

test_loss, test_acc, test_f1, y_true, y_pred = evaluate(model, test_loader, desc="test")

print("Test loss:", test_loss)
print("Test accuracy:", test_acc)
print("Test macro F1:", test_f1)
print()
print(classification_report(y_true, y_pred, target_names=display_class_names, zero_division=0))

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))

plt.figure(figsize=(8, 7))
plt.imshow(cm, cmap="Blues")
plt.title("Regularized ViT Confusion Matrix - 6 Classes")
plt.xticks(range(num_classes), display_class_names, rotation=45, ha="right")
plt.yticks(range(num_classes), display_class_names)
plt.xlabel("Predicted")
plt.ylabel("True")

for i in range(num_classes):
    for j in range(num_classes):
        color = "white" if cm[i, j] > cm.max() * 0.55 else "black"
        plt.text(j, i, str(cm[i, j]), ha="center", va="center", color=color)

plt.colorbar()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "vit_regularized_confusion_matrix.png", dpi=180)
plt.show()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

@torch.no_grad()
def predict_image(image_path, top_k=3):
    image = Image.open(image_path).convert("RGB")
    tensor = eval_transform(image).unsqueeze(0).to(device)
    outputs = model(tensor)
    probs = torch.softmax(outputs, dim=1).squeeze(0).cpu()
    scores, indices = torch.topk(probs, k=min(top_k, num_classes))
    return [(display_class_names[idx], float(score)) for score, idx in zip(scores, indices)]


example_path = test_samples[0][0]
print("Example image:", example_path)
predict_image(example_path)